# Distribution System State Estimation Using Wavelet Decomposition with NFPP Sodium-Ion BESS Performance Evaluation

This notebook implements the complete research pipeline for the DFN-based BESS optimization, dynamic transient simulation under partial observability, and rigorous wavelet-domain statistical state estimation.

In [ ]:
import os
import sys

# Environment Setup and Namespace Package Support
root_dir = os.path.abspath(os.getcwd())
while root_dir and not any(os.path.exists(os.path.join(root_dir, d)) for d in ['nfpp_sodium_ion', 'docs']):
    parent = os.path.dirname(root_dir)
    if parent == root_dir:
        break
    root_dir = parent

if root_dir and os.path.exists(root_dir):
    nfpp_dir = os.path.join(root_dir, 'nfpp_sodium_ion')
    src_dir = os.path.join(root_dir, 'src')
    for p in [root_dir, nfpp_dir]:
        if p not in sys.path:
            sys.path.insert(0, p)
    import src
    if hasattr(src, '__path__') and src_dir not in src.__path__:
        src.__path__.append(src_dir)

import numpy as np
import matplotlib.pyplot as plt
print("Environment initialized successfully.")

## Stage 2: Cell Optimization
Hierarchical Material Discovery + Structural Sensitivity Optimization.

In [ ]:
from src.cell_optimization.parameter_opts import HierarchicalOptimizer

print("Stage 2: Running Hierarchical Material & Structural Optimization...")
optimizer = HierarchicalOptimizer()
optimized_res = optimizer.run()

print("\n--- OPTIMIZATION RESULTS ---")
print("Optimized Design Variables per Objective:")
for obj, specs in optimized_res.get("opt_designs_per_objective", {}).items():
    print(f"\nObjective: {obj.capitalize()}")
    for k, v in specs.items():
        print(f"  {k:40s}: {v:12.6e}")

print("\nSelected Integrated Design Variables:")
for k, v in optimized_res.get("design_specs_representative", {}).items():
    print(f"  {k:40s}: {v:12.6e}")

print("\n--- OPTIMAL CANDIDATE: QM DATA & DERIVED CELL PARAMETERS ---")
mats = optimized_res.get("materials", {})
deltas = optimized_res.get("combined_deltas_representative", {})
for cat in ["cathode", "electrolyte"]:
    print(f"\n{cat.capitalize()} Material:")
    m_data = mats.get(cat, {})
    print(f"  Name: {m_data.get('name') or m_data.get('salt')}")
    print(f"  Formula: {m_data.get('formula')}")
    print("  QM/Physics Properties:")
    for pk, pv in m_data.get("properties", {}).items():
        print(f"    {pk:25s}: {pv}")

print("\nMapping to PyBaMM Parameter Deltas:")
for category, props in deltas.items():
    print(f"  [{category.upper()}]")
    for pk, pv in props.items():
        print(f"    {pk:45s}: {pv:+.4e}")

## Stage 3: Stability Validation & Parameter Extraction
Performance evaluation and resistance profile generation for the digital twin.

In [ ]:
from src.cell_optimization.validate import OptimizationValidator

print("Stage 3: Running Stability Validation...")

design_specs = optimized_res.get("design_specs_representative", {})
deltas = optimized_res.get("combined_deltas_representative", {})

validator = OptimizationValidator(design_specs, deltas, engine=optimizer.engine)
results = validator.run_validation()

print("\nStage 3.1: Running BESS Robustness Evaluation...")
from src.simulation.tests import BESSEvaluator
bess_evaluator = BESSEvaluator(optimized_res)
envelope_res = bess_evaluator.evaluate_bess_performance()

import pandas as pd
from IPython.display import display, HTML

metrics_meta = [
    ("round_trip_energy_efficiency", "Round-Trip Energy Efficiency (RTE)", "eta_RTE", "{:.2%}"),
    ("coulombic_efficiency", "Coulombic Efficiency", "eta_C", "{:.2%}"),
    ("voltage_efficiency", "Voltage Efficiency", "eta_V", "{:.2%}"),
    ("usable_energy_capacity_wh", "Usable Energy Capacity", "E_usable", "{:.2f} Wh"),
    ("power_capability_w", "Power Capability", "P_max", "{:.2f} W"),
    ("thermal_response_delta_t", "Thermal Response Delta T", "Delta T", "{:.2f} K"),
    ("max_temperature_k", "Maximum Temperature", "T_max", "{:.2f} K"),
    ("depth_of_discharge", "Depth of Discharge", "DoD", "{:.2%}"),
    ("equivalent_full_cycles", "Equivalent Full Cycles", "EFC", "{:.4f}"),
    ("capacity_fade", "Capacity Fade", "F_Q", "{:.4e}"),
    ("cycle_life", "Estimated Cycle Life", "N_life", "{:.0f} cycles"),
    ("calendar_life_years", "Estimated Calendar Life", "t_life", "{:.1f} years"),
    ("levelized_cost_of_storage_usd_per_kwh", "Levelized Cost of Storage", "LCOS", "${:.4f}/kWh")
]

rows = []
for key, desc, sym, fmt in metrics_meta:
    val = envelope_res.get(key, 0.0)
    rows.append({"Metric": desc, "Symbol": sym, "Value": fmt.format(val)})

df_metrics = pd.DataFrame(rows)
display(HTML("<h3>NFPP BESS Robustness Evaluation Framework Metrics (paper.md aligned)</h3>"))
display(df_metrics)

## Stage 4: Wavelet-Domain Observability Datasets under Partial Observability
This section generates the two distinct decoupled datasets (Dataset 1 and Dataset 2) using OpenDSS and verifies the partial PCC metering architecture on the hidden network.

In [ ]:
from src.simulation.dataset import generate_experiments_dataset
import pandas as pd

print("Stage 4.1: Generating Decoupled Observability Datasets...")
dataset_1, dataset_2 = generate_experiments_dataset(n_scenarios=15, write_to_disk=False)
print("Datasets generated successfully.")

In [ ]:
from IPython.display import display, HTML
import pandas as pd

print("Stage 4.2: Tabulating Dataset 1 (Scenario-Based Dataset with Line Parameters and ONLY Transformer Steady State Readings)")

rows_1 = []
for item in dataset_1:
    gt = item["ground_truth"]
    obs = item["observations"]["features"]
    f_id = gt["feeder_id"]
    f_num = f_id.split("_")[-1]
    pcc_id = f"trans{f_num}_lv_pcc"
    row = {
        "Scenario": gt["scenario_id"],
        "Feeder ID": f_id,
        "Topology": gt["topology_type"],
        "Line Mult": gt["line_parameter_multiplier"],
        "Total Buses": gt["hidden_total_buses"],
        "Total Edges": gt["hidden_total_edges"],
        "Trans V_avg (LV)": obs.get(f"{pcc_id}_voltage_mag_avg", 0.0),
        "Trans Current_avg": obs.get(f"{pcc_id}_current_mag_avg", 0.0),
        "Trans P (kW)": obs.get(f"{pcc_id}_p_kw", 0.0),
        "Trans Q (kVAR)": obs.get(f"{pcc_id}_q_kvar", 0.0)
    }
    rows_1.append(row)
    
df_1 = pd.DataFrame(rows_1)
display(HTML("<h3>Dataset 1 (Transformer Steady State & Line Parameters)</h3>"))
display(df_1.head(15))

In [ ]:
print("Stage 4.3: Tabulating Dataset 2 (Event-Based Wavelet-Domain Transient Dataset with Normalization and Decomposition)")

rows_2 = []
for item in dataset_2:
    gt = item["ground_truth"]
    obs_info = item["observations"]
    features = obs_info["features"]
    
    pcc_id = obs_info["pcc_id"]
    v_0_cD1_std = features.get(f"{pcc_id}_v_0_cD1_std", 0.0)
    v_0_cD2_energy = features.get(f"{pcc_id}_v_0_cD2_energy", 0.0)
    
    row = {
        "Scenario": gt["scenario_id"],
        "Feeder ID": gt["feeder_id"],
        "PCC ID": pcc_id,
        "Event": gt["simulated_event"],
        "Parent Trans V_ss Ref": obs_info["steady_state_reference"]["v_mags_ss"][0],
        "Phase A cD1 std": v_0_cD1_std,
        "Phase A cD2 Energy": v_0_cD2_energy
    }
    rows_2.append(row)
    
df_2 = pd.DataFrame(rows_2)
display(HTML("<h3>Dataset 2 (Wavelet-Domain Transient Observations)</h3>"))
display(df_2.head(15))

## Stage 5: Rigorous Statistical Validation Pipeline on Actual Wavelet Representations
This section executes the rigorous non-parametric statistical tests directly on the actual generated Dataset 1 and Dataset 2.

In [ ]:
from src.statistics.dependence import permutation_test_dcor, benjamini_hochberg_correction
import numpy as np

print("Test 1: Distance Correlation (Exploratory Multiple testing with BH FDR Correction)")

X = []
Y_list = []
for item in dataset_1:
    gt = item["ground_truth"]
    obs = item["observations"]["features"]
    f_num = gt["feeder_id"].split("_")[-1]
    pcc_id = f"trans{f_num}_lv_pcc"
    X.append([gt["line_parameter_multiplier"], gt["hidden_total_buses"]])
    Y_list.append([
        obs.get(f"{pcc_id}_voltage_mag_avg", 0.0),
        obs.get(f"{pcc_id}_current_mag_avg", 0.0),
        obs.get(f"{pcc_id}_p_kw", 0.0)
    ])
X = np.array(X)
Y = np.array(Y_list)

res_dcor = permutation_test_dcor(X, Y, n_permutations=99, seed=42)
raw_p = res_dcor["p_value"]
adjusted_p = benjamini_hochberg_correction([raw_p])[0]

print(f"Distance Correlation Statistic: {res_dcor['statistic']:.4f}")
print(f"Raw Permutation p-value:        {raw_p:.4f}")
print(f"Adjusted p-value (FDR):         {adjusted_p:.4f}")

In [ ]:
from src.statistics.dependence import permutation_test_hsic

print("Test 1.1: Hilbert-Schmidt Independence Criterion (HSIC) Nonlinear Confirmation Test")
res_hsic = permutation_test_hsic(X, Y, n_permutations=99, seed=42)
print(f"HSIC Statistic:        {res_hsic['statistic']:.6f}")
print(f"HSIC Permutation p-val: {res_hsic['p_value']:.4f}")

In [ ]:
from src.statistics.distribution import permutation_test_mmd

print("Test 2: Maximum Mean Discrepancy (MMD) Two-Sample Test (Radial vs Ring Distribution Separation)")

Y_radial = []
Y_ring = []
for i, item in enumerate(dataset_1):
    topo_type = item["ground_truth"]["topology_type"]
    y_val = Y[i]
    if topo_type == "radial":
        Y_radial.append(y_val)
    else:
        Y_ring.append(y_val)

Y_radial = np.array(Y_radial)
Y_ring = np.array(Y_ring)

if len(Y_ring) > 0 and len(Y_radial) > 0:
    res_mmd = permutation_test_mmd(Y_radial, Y_ring, n_permutations=99, seed=42)
    print(f"MMD^2 Statistic:       {res_mmd['statistic']:.6f}")
    print(f"MMD Permutation p-val: {res_mmd['p_value']:.4f}")
else:
    print("Skip: Not enough samples for both Radial and Ring topologies to execute MMD test.")

In [ ]:
from src.statistics.permanova import permanova
from src.statistics.dispersion import dispersion_test

print("Test 3: PERMANOVA & Multivariate Dispersion Analysis (PERMDISP) Group Homogeneity")

Y_wavelet_list = []
groups = []
for item in dataset_2:
    features = item["observations"]["features"]
    pcc_id = item["observations"]["pcc_id"]
    groups.append(item["ground_truth"]["simulated_event"])
    Y_wavelet_list.append([
        features.get(f"{pcc_id}_v_0_cD1_std", 0.0),
        features.get(f"{pcc_id}_v_0_cD2_std", 0.0),
        features.get(f"{pcc_id}_v_0_cD1_energy", 0.0),
        features.get(f"{pcc_id}_v_0_cD2_energy", 0.0)
    ])
Y_wavelet = np.array(Y_wavelet_list)

res_perm = permanova(Y_wavelet, groups, n_permutations=99, seed=42)
print(f"PERMANOVA F-pseudo:   {res_perm['F_pseudo']:.4f}")
print(f"PERMANOVA p-value:    {res_perm['p_value']:.4f}")
print(f"PERMANOVA R2:         {res_perm['r_squared']:.4f}")

res_disp = dispersion_test(Y_wavelet, groups, n_permutations=99, seed=42)
print(f"\nDispersion F-stat:    {res_disp['F_dispersion']:.4f}")
print(f"Dispersion p-value:   {res_disp['p_value']:.4f}")

In [ ]:
from src.statistics.equivalence import tost_equivalence

print("Test 4: TOST Practical Equivalence Testing on actual Wavelet representations")

y_a_wavelet = []
y_b_wavelet = []
for item in dataset_2:
    event = item["ground_truth"]["simulated_event"]
    pcc_id = item["observations"]["pcc_id"]
    val = item["observations"]["features"].get(f"{pcc_id}_v_0_cD1_std", 0.0)
    if event == "transformer_inrush":
        y_a_wavelet.append(val)
    elif event == "capacitor_switching":
        y_b_wavelet.append(val)

if len(y_a_wavelet) > 1 and len(y_b_wavelet) > 1:
    res_tost = tost_equivalence(y_a_wavelet, y_b_wavelet, margin=0.15)
    print(f"Mean Diff:            {res_tost['difference']:.6f}")
    print(f"Equivalence Margin:   {res_tost['margin']:.4f}")
    print(f"TOST p-value:         {res_tost['p_equivalence']:.4f}")
    print(f"Practically Equiv?:   {res_tost['equivalent']}")
else:
    print("Skip: Not enough samples for TOST equivalence testing.")

In [ ]:
from src.features.wavelet_processor import process_pcc_waveforms
from src.statistics.dependence import distance_correlation

print("Test 5: Waveform Noise Robustness Sweep (SNR: 40, 30, 20, 10, 5 dB applied directly to actual waveforms)")

snr_levels = [40, 30, 20, 10, 5]
for snr in snr_levels:
    noisy_Y_features = []
    
    for item in dataset_2:
        obs = item["observations"]
        pcc_id = obs["pcc_id"]
        time = np.array(obs["raw_transient_waveform"]["time"])
        raw_v = np.array(obs["raw_transient_waveform"]["voltage_abc"])
        raw_i = np.array(obs["raw_transient_waveform"]["current_abc"])
        
        rng_noise = np.random.default_rng(42)
        
        v_power = np.mean(raw_v**2)
        v_noise_std = np.sqrt(v_power / (10 ** (snr / 10.0)))
        noisy_v = raw_v + rng_noise.normal(0.0, v_noise_std, size=raw_v.shape)
        
        i_power = np.mean(raw_i**2)
        i_noise_std = np.sqrt(i_power / (10 ** (snr / 10.0)))
        noisy_i = raw_i + rng_noise.normal(0.0, i_noise_std, size=raw_i.shape)
        
        processed = process_pcc_waveforms(pcc_id, time, noisy_v, noisy_i, event_start=0.02)
        
        noisy_Y_features.append([
            processed.features.get(f"{pcc_id}_v_0_cD1_std", 0.0),
            processed.features.get(f"{pcc_id}_v_0_cD2_std", 0.0),
            processed.features.get(f"{pcc_id}_v_0_cD1_energy", 0.0),
            processed.features.get(f"{pcc_id}_v_0_cD2_energy", 0.0)
        ])
        
    noisy_Y = np.array(noisy_Y_features)
    
    X_dcor = []
    for item in dataset_2:
        scen_id = item["ground_truth"]["scenario_id"]
        f_id = item["ground_truth"]["feeder_id"]
        # Join exactly by scenario_id and feeder_id
        for gt_1_item in dataset_1:
            gt_info = gt_1_item["ground_truth"]
            if gt_info["scenario_id"].startswith(scen_id) and gt_info["feeder_id"] == f_id:
                X_dcor.append(gt_info["line_parameter_multiplier"])
                break
    X_dcor = np.array(X_dcor)
    
    dcor_val = distance_correlation(X_dcor, noisy_Y)
    print(f"SNR = {snr:2d} dB | Distance Correlation (hidden multiplier vs noisy SWT features): {dcor_val:.4f}")